# Mimicry Evaluation on Colab

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

In [ ]:
from pathlib import Path

REPO_URL = "YOUR_REPO_URL"
PROJECT_DIR = Path('/content/Mimicry')
DRIVE_ROOT = Path('/content/drive/MyDrive/PyWatermark')

TEST_DIR = DRIVE_ROOT / 'datasets' / 'test'
CHECKPOINT_PATH = DRIVE_ROOT / 'artifacts' / 'checkpoints_robust_16' / 'best.pt'
REPORT_DIR = DRIVE_ROOT / 'artifacts' / 'reports_robust_16'
HISTORY_PATH = DRIVE_ROOT / 'artifacts' / 'logs_robust_16' / 'metrics_history.csv'
PLOTS_DIR = DRIVE_ROOT / 'artifacts' / 'plots_robust_16'

## 3. Clone or update the repo

In [ ]:
if PROJECT_DIR.exists():
    %cd {PROJECT_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} Mimicry
    %cd {PROJECT_DIR}

!pip install -q -r requirements.txt

## 4. Check files

In [ ]:
print('test dir exists:', TEST_DIR.exists())
print('checkpoint exists:', CHECKPOINT_PATH.exists())
print('history exists:', HISTORY_PATH.exists())

## 5. Evaluate checkpoint

In [ ]:
%cd {PROJECT_DIR}
!python -m evaluation.evaluate \
    --test-dir {TEST_DIR} \
    --checkpoint {CHECKPOINT_PATH} \
    --report-dir {REPORT_DIR} \
    --batch-size 16 \
    --num-workers 2

## 6. Show report

In [ ]:
REPORT_PATH = REPORT_DIR / 'evaluation_report.txt'
report_text = REPORT_PATH.read_text(encoding='utf-8')
print(report_text)

## 7. Compact summary

In [ ]:
import re
from IPython.display import Markdown, display

def extract_metric(pattern, text, default='N/A'):
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1) if match else default

summary = f'''### Final Evaluation Summary
- Checkpoint: `{CHECKPOINT_PATH}`
- PSNR: `{extract_metric(r'PSNR: ([0-9.]+)', report_text)} dB`
- SSIM: `{extract_metric(r'SSIM: ([0-9.]+)', report_text)}`
- Clean bit accuracy: `{extract_metric(r'^clean\s+([0-9.]+)', report_text)}`
- JPEG bit accuracy: `{extract_metric(r'^jpeg\s+([0-9.]+)', report_text)}`
- Blur bit accuracy: `{extract_metric(r'^blur\s+([0-9.]+)', report_text)}`
- Crop bit accuracy: `{extract_metric(r'^crop\s+([0-9.]+)', report_text)}`
- Noise bit accuracy: `{extract_metric(r'^noise\s+([0-9.]+)', report_text)}`
- Brightness bit accuracy: `{extract_metric(r'^brightness\s+([0-9.]+)', report_text)}`
'''
display(Markdown(summary))

## 8. Export plots

In [ ]:
%cd {PROJECT_DIR}
!python -m evaluation.plot_training_curves \
    --history {HISTORY_PATH} \
    --output-dir {PLOTS_DIR} \
    --title-prefix "Mimicry robust_16"

## 9. List outputs

In [ ]:
!ls {REPORT_DIR}
!ls {PLOTS_DIR}